# Preparação e auditoria dos dados

Este notebook repara a **estrutura** de `train.csv`, documenta cada alteração e
produz um dataset que pode ser relido com um parser CSV estrito.

Os campos clínicos disponíveis, a ordem dos casos e as especialidades são preservados.
Textos repetidos, rótulos diferentes e campos vazios são **assinalados**, sem eliminar
exemplos nem escolher rótulos. O ficheiro `test_no_labels.csv` é usado apenas para
verificar o seu hash antes e depois; o conteúdo não entra nesta análise.

As funções estão em `data_preparation.py`. As regras e os resultados são discutidos
abaixo. Executar todas as células, desde um kernel novo, reproduz os outputs.


## 1. Ambiente e ficheiros

Antes da primeira execução, na pasta do projeto:

```powershell
python -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
.\.venv\Scripts\python.exe -m jupyter lab
```

Selecionar o kernel de `.venv` e executar **Restart Kernel and Run All Cells**.
Não é necessário editar caminhos absolutos neste notebook.

Referências das ferramentas: [Jupyter](https://jupyter.org/install) e
[execução de notebooks com NBClient](https://nbclient.readthedocs.io/en/latest/client.html).


In [1]:
import csv
import json
import platform
import sys
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# Funciona com o kernel iniciado na raiz, numa subpasta ou na pasta LN.
locations = []
for parent in (Path.cwd(), *Path.cwd().parents):
    locations.extend([parent, parent / 'help-the-doctors'])
ROOT = next((p for p in locations if (p / 'train.csv').is_file()
             and (p / 'data_preparation.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abre o notebook na pasta do projeto que contém train.csv.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data_preparation import (
    COLUMNS, SPECIALTIES, audit_records, export_reader, export_results,
    repair_training_csv, sha256,
)

SOURCE = ROOT / 'train.csv'
TEST_SOURCE = ROOT / 'test_no_labels.csv'
original_hash = sha256(SOURCE)
test_hash = sha256(TEST_SOURCE)
environment = {'python': platform.python_version(),
               **{package: version(package) for package in ['pandas', 'ipykernel', 'nbclient', 'jupyterlab']}}
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 40)
display(pd.Series(environment, name='Versão').to_frame())


,Versão
python,3.14.3
pandas,3.0.6
ipykernel,7.4.0
nbclient,0.11.0
jupyterlab,4.6.4


## 2. O que acontece numa leitura CSV direta?

Uma aspa de abertura sem fecho pode fazer o parser consumir as linhas seguintes
como parte do mesmo campo. Ter cinco colunas no resultado não prova que os casos
foram separados corretamente. Comparamos a leitura direta com o número de linhas
que começam por uma das 12 especialidades.


In [2]:
with SOURCE.open(encoding='utf-8-sig', newline='') as stream:
    raw_reader = csv.DictReader(stream, delimiter=';')
    raw_records = list(raw_reader)
with SOURCE.open(encoding='utf-8-sig', newline='') as stream:
    physical_lines = stream.readlines()

physical_case_count = sum(line.partition(';')[0] in SPECIALTIES for line in physical_lines[1:])
raw_invalid_count = sum(row['medical_specialty'] not in SPECIALTIES for row in raw_records)
display(pd.Series({
    'Linhas físicas de dados': len(physical_lines) - 1,
    'Linhas que começam por uma especialidade': physical_case_count,
    'Registos obtidos por leitura direta': len(raw_records),
    'Rótulos não reconhecidos na leitura direta': raw_invalid_count,
}, name='Contagem').to_frame())


,Contagem
Linhas físicas de dados,2672
Linhas que começam por uma especialidade,2669
Registos obtidos por leitura direta,2616
Rótulos não reconhecidos na leitura direta,3


## 3. Regras de reparação

1. **Descrição truncada:** quando uma linha tem uma especialidade válida, um único
   `;` e uma única aspa de abertura na descrição, fecha-se essa aspa. A descrição
   disponível é preservada; `sample_name`, `transcription` e `keywords` ficam vazios.
2. **Colunas excedentes:** quando existem sete campos e os dois últimos estão vazios,
   conservam-se os primeiros cinco. Não se descarta conteúdo de campos excedentes.
3. **Continuação de transcrição:** uma linha sem especialidade, com texto apenas no
   primeiro campo, é ligada ao caso anterior apenas quando este acabou de ter
   colunas vazias excedentes removidas. Insere-se uma quebra de linha e preserva-se
   o texto disponível. Não se reconstroem letras ou palavras perdidas.
4. **Possível truncagem:** um caso reparado cuja transcrição termina num cabeçalho
   com dois-pontos é assinalado para revisão. Esta é uma indicação, não uma certeza.

Esta leitura por linha física aplica-se à estrutura do ficheiro fornecido. Um
CSV diferente com campos multilinha legítimos necessita de regras próprias.
Estruturas desconhecidas geram um erro com o número da linha.

**Nenhuma decisão de filtragem ou correção clínica é tomada nesta etapa.**


In [3]:
records, changes = repair_training_csv(SOURCE)
audit = audit_records(records)
clean_df = pd.DataFrame([{field: row[field] for field in COLUMNS} for row in records])
audit_df = pd.DataFrame(audit)
changes_df = pd.DataFrame(changes, columns=['record_id', 'source_line', 'action', 'details'])

display(changes_df)


,record_id,source_line,action,details
0,160,161,close_truncated_description,Fechada a aspa da descrição e representados como vazios os três campos indisponíveis: sample_nam...
1,162,163,remove_extra_empty_fields,Removidos apenas os dois campos vazios excedentes após os cinco campos esperados.
2,713,714,close_truncated_description,Fechada a aspa da descrição e representados como vazios os três campos indisponíveis: sample_nam...
3,946,947,close_truncated_description,Fechada a aspa da descrição e representados como vazios os três campos indisponíveis: sample_nam...
4,960,961,remove_extra_empty_fields,Removidos apenas os dois campos vazios excedentes após os cinco campos esperados.
5,960,962,join_transcription_continuation,Continuação ligada ao caso anterior com uma quebra de linha. Preservados os caracteres disponíve...
6,1086,1088,close_truncated_description,Fechada a aspa da descrição e representados como vazios os três campos indisponíveis: sample_nam...
7,1094,1096,remove_extra_empty_fields,Removidos apenas os dois campos vazios excedentes após os cinco campos esperados.
8,1094,1096,flag_possible_truncation,A transcrição termina num cabeçalho com dois-pontos; possível texto truncado na origem.
9,2087,2089,close_truncated_description,Fechada a aspa da descrição e representados como vazios os três campos indisponíveis: sample_nam...


## 4. Verificações de integridade

Cada linha física tem de estar representada exatamente uma vez: como início de
um caso ou como continuação. Os rótulos e a ordem têm de coincidir com os casos
identificáveis na origem. Estas verificações não dependem de números fixos de registo.


In [4]:
original_labels = [line.partition(';')[0] for line in physical_lines[1:]
                   if line.partition(';')[0] in SPECIALTIES]
represented_lines = [line for row in records
                     for line in range(row['source_line_start'], row['source_line_end'] + 1)]
checks = {
    'Cinco campos no dataset reparado': list(clean_df.columns) == COLUMNS,
    'Todos os casos identificáveis recuperados': len(records) == physical_case_count,
    'Especialidades válidas': bool(clean_df['medical_specialty'].isin(SPECIALTIES).all()),
    'As 12 especialidades estão representadas': set(clean_df['medical_specialty']) == set(SPECIALTIES),
    'Rótulos e ordem preservados': clean_df['medical_specialty'].tolist() == original_labels,
    'Cada linha de origem representada uma vez': represented_lines == list(range(2, len(physical_lines) + 1)),
    'CSV original intacto': sha256(SOURCE) == original_hash,
    'Teste oficial intacto': sha256(TEST_SOURCE) == test_hash,
}
display(pd.Series(checks, name='Verificado').to_frame())
assert all(checks.values()), 'Uma verificação de integridade falhou.'

display(pd.DataFrame({
    'Leitura direta': [len(raw_records), raw_invalid_count],
    'Após reparação': [len(records), 0],
}, index=['Registos', 'Rótulos não reconhecidos']))


,Verificado
Cinco campos no dataset reparado,True
Todos os casos identificáveis recuperados,True
Especialidades válidas,True
As 12 especialidades estão representadas,True
Rótulos e ordem preservados,True
Cada linha de origem representada uma vez,True
CSV original intacto,True
Teste oficial intacto,True


,Leitura direta,Após reparação
Registos,2616,2669
Rótulos não reconhecidos,3,0


## 5. Distribuição das especialidades

As contagens seguintes usam os casos separados corretamente. As contagens da
primeira leitura eram inferiores porque algumas linhas ficaram dentro de descrições.
O desequilíbrio não é corrigido removendo classes nesta etapa.


In [5]:
class_counts = clean_df['medical_specialty'].value_counts().rename('Casos').to_frame()
class_counts['Percentagem'] = (100 * class_counts['Casos'] / len(clean_df)).round(2)
class_counts.index.name = 'Especialidade'
display(class_counts)


,Casos,Percentagem
Especialidade,,
Surgery,893,33.46
Cardiovascular-Pulmonary,309,11.58
Orthopedic,300,11.24
Radiology,222,8.32
General Medicine,212,7.94
Gastroenterology,195,7.31
Neurology,184,6.89
Obstetrics-Gynecology,136,5.10
Neurosurgery,79,2.96


## 6. Campos vazios e textos parciais

Uma transcrição vazia pode coexistir com uma descrição útil. Os casos ficam no
dataset, com essa informação na auditoria. Um caso reparado pode continuar
incompleto porque o texto em falta já não existe no ficheiro fornecido.


In [6]:
empty_columns = [field + '_empty' for field in COLUMNS[1:]]
missing_counts = audit_df[empty_columns].sum().rename('Casos com campo vazio').to_frame()
missing_counts.index = COLUMNS[1:]
display(missing_counts)

display(audit_df.loc[audit_df['is_repaired'], [
    'record_id', 'source_line_start', 'source_line_end', 'medical_specialty',
    'sample_name', 'repair_actions', 'is_partial',
]])


,Casos com campo vazio
description,1
sample_name,7
transcription,31
keywords,450


,record_id,source_line_start,source_line_end,medical_specialty,sample_name,repair_actions,is_partial
159,160,161,161,Cardiovascular-Pulmonary,,close_truncated_description,True
161,162,163,163,Psychiatry-Psychology,Neuropsychological Evaluation - 5,remove_extra_empty_fields,False
712,713,714,714,General Medicine,,close_truncated_description,True
945,946,947,947,Neurosurgery,,close_truncated_description,True
959,960,961,962,General Medicine,Dog Bite,remove_extra_empty_fields;join_transcription_continuation,True
1085,1086,1088,1088,Obstetrics-Gynecology,,close_truncated_description,True
1093,1094,1096,1096,General Medicine,Trouble Breathing - H&P,remove_extra_empty_fields;flag_possible_truncation,True
2086,2087,2089,2089,Surgery,,close_truncated_description,True
2097,2098,2100,2101,Neurology,TIA & Lumbar Stenosis,remove_extra_empty_fields;join_transcription_continuation,True
2325,2326,2329,2329,Surgery,,close_truncated_description,True


In [7]:
empty_transcriptions = audit_df.loc[audit_df['transcription_empty'], [
    'record_id', 'source_line_start', 'source_line_end', 'medical_specialty',
    'sample_name', 'description_empty', 'is_partial',
]]
display(empty_transcriptions)
display(audit_df['transcription_characters'].describe().round(1).rename('Caracteres na transcrição').to_frame())


,record_id,source_line_start,source_line_end,medical_specialty,sample_name,description_empty,is_partial
148,149,150,150,Cardiovascular-Pulmonary,Tachypnea,False,False
159,160,161,161,Cardiovascular-Pulmonary,,False,True
385,386,387,387,Gastroenterology,EGD Template - 1,False,False
399,400,401,401,Gastroenterology,Colonoscopy Template - 5,False,False
416,417,418,418,Gastroenterology,Colonoscopy Template - 3,False,False
452,453,454,454,Gastroenterology,Colonoscopy Template - 2,False,False
455,456,457,457,Gastroenterology,EGD Template - 4,False,False
493,494,495,495,Gastroenterology,Colonoscopy Template - 1,False,False
712,713,714,714,General Medicine,,False,True
945,946,947,947,Neurosurgery,,False,True


,Caracteres na transcrição
count,2669.0
mean,2987.1
std,1977.8
min,0.0
25%,1538.0
50%,2595.0
75%,4010.0
max,15194.0


## 7. Textos repetidos e rótulos diferentes

A igualdade usa a transcrição completa, removendo apenas espaços nas extremidades
para formar os grupos. Os textos exportados mantêm-se. Transcrições vazias não
formam grupos de duplicados.

Os casos com o mesmo texto e rótulos diferentes permanecem com os rótulos originais.
Esta ambiguidade deve ser discutida no relatório. Na futura divisão dos dados,
todos os casos de um grupo devem ficar no mesmo conjunto para evitar sobreposição
entre treino e avaliação.


In [8]:
duplicate_groups = audit_df.loc[audit_df['duplicate_group_id'].ne('')].drop_duplicates('duplicate_group_id')
conflicting_groups = duplicate_groups.loc[duplicate_groups['conflicting_labels']]
display(pd.Series({
    'Grupos de transcrições repetidas': len(duplicate_groups),
    'Casos pertencentes a esses grupos': int(audit_df['duplicate_group_id'].ne('').sum()),
    'Grupos com especialidades diferentes': len(conflicting_groups),
    'Casos em grupos com especialidades diferentes': int(audit_df['conflicting_labels'].sum()),
}, name='Contagem').to_frame())

display(conflicting_groups[[
    'sample_name', 'duplicate_record_ids', 'group_labels',
]].head(10))


,Contagem
Grupos de transcrições repetidas,731
Casos pertencentes a esses grupos,1530
Grupos com especialidades diferentes,730
Casos em grupos com especialidades diferentes,1528


,sample_name,duplicate_record_ids,group_labels
3,Lower Extremity Venous Doppler,4;1716,Cardiovascular-Pulmonary;Radiology
8,Ultrasound - Carotid - 1,9;1718,Cardiovascular-Pulmonary;Radiology
9,CT Angiography,10;1759,Cardiovascular-Pulmonary;Radiology
11,Cardiac Catheterization - 4,12;1945,Cardiovascular-Pulmonary;Surgery
12,"Heart Catheterization, Ventriculography, & Angiography",13;2171,Cardiovascular-Pulmonary;Surgery
14,Heart Catheterization & Angiography - 2,15;2523,Cardiovascular-Pulmonary;Surgery
15,"Heart Catheterization, Ventriculography, & Angiography - 2",16;2174,Cardiovascular-Pulmonary;Surgery
16,Stenting,17;2646,Cardiovascular-Pulmonary;Surgery
17,2-D Echocardiogram - 2,18;1709,Cardiovascular-Pulmonary;Radiology
19,Patent Ductus Arteriosus,20;2031,Cardiovascular-Pulmonary;Surgery


## 8. Inspeção de casos concretos

Exemplo: o caso da linha 961 e a continuação da linha 962 passam a ser um só registo.
O texto mantém a fronteira entre as partes e os caracteres disponíveis. A reparação
da estrutura não garante que a nota clínica esteja completa.

Mudar `example_line` permite inspecionar outro caso usando a linha original.


In [9]:
example_line = 961
example = next(row for row in records
               if row['source_line_start'] <= example_line <= row['source_line_end'])
display(pd.Series({
    'Registo novo': example['record_id'],
    'Linha inicial original': example['source_line_start'],
    'Linha final original': example['source_line_end'],
    'Especialidade': example['medical_specialty'],
    'Procedimento': example['sample_name'],
    'Texto parcial': example['is_partial'],
}, name='Caso').to_frame())
print(example['transcription'][:1200])


,Caso
Registo novo,960
Linha inicial original,961
Linha final original,962
Especialidade,General Medicine
Procedimento,Dog Bite
Texto parcial,True


CHIEF COMPLAINT:  Dog bite to his right lower leg.,HISTORY OF PRESENT ILLNESS:  This 50-year-old white male earlier this afternoon was attempting to adjust a cable that a dog was tied to.  Dog was a German shepherd, it belonged to his brother, and the dog spontaneously attacked him.  He sustained a bite to his right lower leg.  Apparently, according to the patient, the dog is well known and is up-to-date on his shots and they wanted to confirm that.  The dog has given no prior history of any reason to believe he is not a healthy dog.  The patient himself developed a puncture wound with a flap injury.  The patient has a flap wound also below the puncture wound, a V-shaped flap, which is pointing towards the foot.  It appears to be viable.  The wou"
d is open about may be roughly a centimeter in the inside of the flap.  He was seen by his medical primary care physician and was given a tetanus shot and the wound was cleaned and wrapped, and then he was referred to us for further assessmen

## 9. Exportar e reler

`train_clean.csv` conserva exclusivamente os cinco campos do dataset. A auditoria
contém os identificadores novos, as linhas originais, as alterações e os grupos de
duplicados. Os CSVs usam `;` e UTF-8 com BOM.

A exportação é seguida de uma releitura estrita e comparação de **todos** os campos.
Executar novamente substitui apenas os outputs gerados. Os originais são preservados.


In [10]:
summary = export_results(ROOT, records, changes, audit)
summary['environment'] = environment
(ROOT / 'outputs' / 'auditoria' / 'summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
assert sha256(SOURCE) == original_hash
assert sha256(TEST_SOURCE) == test_hash

display(pd.Series({
    'Casos exportados': summary['records'],
    'Casos reparados': summary['repaired_records'],
    'Casos parciais ou potencialmente truncados': summary['partial_records'],
    'Transcrições vazias': summary['empty_fields']['transcription'],
    'Grupos com rótulos diferentes': summary['conflicting_label_groups'],
}, name='Resultado').to_frame())


,Resultado
Casos exportados,2669
Casos reparados,13
Casos parciais ou potencialmente truncados,11
Transcrições vazias,31
Grupos com rótulos diferentes,730


## 10. Leitor do dataset reparado

O novo leitor mostra os casos reparados e as linhas do CSV original. As anotações
usam o hash do dataset reparado, para não misturar os identificadores novos com os
do leitor anterior. O ficheiro anterior é mantido como referência da primeira leitura.


In [11]:
reader_path = export_reader(ROOT, records, audit, summary)
outputs = [
    ROOT / 'data' / 'processed' / 'train_clean.csv',
    ROOT / 'outputs' / 'auditoria' / 'repairs.csv',
    ROOT / 'outputs' / 'auditoria' / 'record_audit.csv',
    ROOT / 'outputs' / 'auditoria' / 'summary.json',
    reader_path,
]
display(pd.DataFrame([{
    'Ficheiro': path.relative_to(ROOT).as_posix(),
    'Tamanho (KB)': round(path.stat().st_size / 1024, 1),
} for path in outputs]))
display(Markdown('Abrir no browser: [leitor do dataset reparado](outputs/dados-legiveis/ler_dataset_clean.html).'))


,Ficheiro,Tamanho (KB)
0,data/processed/train_clean.csv,8780.9
1,outputs/auditoria/repairs.csv,2.6
2,outputs/auditoria/record_audit.csv,584.4
3,outputs/auditoria/summary.json,1.3
4,outputs/dados-legiveis/ler_dataset_clean.html,9513.6


Abrir no browser: [leitor do dataset reparado](outputs/dados-legiveis/ler_dataset_clean.html).

## Decisões que ficam para a etapa seguinte

- Definir quais os campos usados por cada modelo e como tratar entradas incompletas.
- Criar um teste interno de cerca de 20%, mantendo os grupos de textos repetidos
  juntos e procurando preservar as proporções das classes.
- Validar modelos nos restantes dados. Ajustar TF-IDF e fazer data augmentation
  apenas no treino de cada divisão.

Este notebook termina na reparação e auditoria. Não cria splits nem usa o teste
oficial para treinar ou escolher modelos.
